<center>
    <img src="https://avatars.githubusercontent.com/u/192127829?s=100" alt="Logo Pyvoa" /> 
</center>

# Pyvoa Front Stress Test
Tristan Beau

We test here lot of features of pyvoa. This may also be seen as a demonstrator of what pyvoa can provide.

*Caution :* This notebook requires lot of ressources, esp. to the browser memory, due to graphical outputs. You may need to execute cell by cell, clearing cell outputs when it works.

To run every case at once, use the script twin of this notebook, `examples/pyfiles/stress_test.py`: it writes each chart to a file instead of displaying it, and gathers them in an `index.html` page with the status of every case.

In [ ]:
# Run once on Colab / Binder, or on a fresh environment.
# %pip install pyvoa

In [ ]:
from IPython.display import Markdown
import random
import pyvoa.front as pf
from pyvoa.tools import PyvoaError
pf.listvis()

In [ ]:
pf.listwhom(True)

In [ ]:
# Fast test of db
l_pbl_db=[]
l_pbl_e=[]
for db in pf.listwhom():
    pf.setvis('bokeh')
    display(Markdown('# '+db))
    try:
        if db != pf.getwhom(None,False,False):
            pf.setwhom(db,reload=True)
        if type(pf.listwhere())==list:
            w=random.sample(pf.listwhere(),k=4)
            print(w)
            try:
                pf.plot(where=w)
            except PyvoaError:  #hypothesis no data for those locations
                pf.plot()
        else:
            pf.plot()
        pf.hist()
        pf.map()
        pf.setvis('matplotlib')
        pf.map()
    except Exception as e:
        l_pbl_db.append(db)
        l_pbl_e.append(e)
        continue
print("Problem with those db =",l_pbl_db," , with error or exception = ",l_pbl_e)

In [ ]:
# Testing regions of the world
import pyvoa.geo as pg
pf.setvis('matplotlib')
pf.setwhom('jhu',reload=False)

gr=pg.GeoRegion()
for r in gr.get_region_list():
    print(r)
    pf.plot(where=r)

In [ ]:
# quick test of all type of plots: each chart, maps on every tile included, is
# drawn under matplotlib then under bokeh with the same arguments, so that the
# two come one under the other; a chart one backend does not offer (compare and
# spiral under matplotlib) is drawn by the other alone
pf.setwhom('owid',reload=False)            # Our World in Data, worldwide, by country
common = dict(which='total_deaths', where='Western Europe', what='daily', option='smooth7')
dated = dict(common, when='20/02/2022')
vis_list = ['matplotlib', 'bokeh']
pf.setvis(vis_list[0])                     # the list*() functions need a backend
cases = [('hist', dict(dated, typeofhist=h)) for h in pf.listhist()]
for m in pf.listmap():
    if m == 'dense':                       # a dense map is drawn without tiles
        cases.append(('map', dict(dated, typeofmap=m)))
    else:
        cases += [('map', dict(dated, typeofmap=m, tile=t)) for t in pf.listtile()]
plots = []
for v in vis_list:
    pf.setvis(v)
    plots += [p for p in pf.listplot() if p not in plots]
for p in plots:
    if p == 'versus':
        kw = dict(common, which=['total_deaths','cur_hosp_patients'])
    elif p in ('spiral', 'yearly'):
        kw = dict(common, where='france')
    else:
        kw = dict(common)
    cases.append(('plot', dict(kw, typeofplot=p)))

for function, kw in cases:
    for v in vis_list:
        pf.setvis(v)
        if kw['typeof'+function] in getattr(pf, 'list'+function)():
            print(v, function, kw)
            getattr(pf, function)(**kw)

In [ ]:
pf.set_verbose_mode(1)
# Advanced outputs test for just one base
pf.setwhom('mpoxgh',reload=False) # not too many which to test 
ww='Brazil'
for v in pf.listvis():
    if v=='bokeh':
        continue
    pf.setvis(v)
    display(Markdown('# Visu = '+v))
    for whi in pf.listwhich():
        for wha in pf.listwhat():
            lo=pf.listoption()
            lo.append('')
            for opt in lo:
                if opt=='nonneg' and whi[0:3]=='cur':
                    continue
                if opt=='nonneg':
                    continue
                for plo in pf.listplot():
                    if plo in ['spiral','menulocation'] and v=='matplotlib':
                        continue
                    if plo in ['versus']:
                        continue
                    if plo == 'yearly' and opt!='sumall':
                        continue
                    print(whi,wha,opt,plo,"plot")
                    if plo in ['spiral','yearly']:
                        pf.plot(which=whi,what=wha,option=opt,typeofplot=plo,where=ww)
                    else:
                        if opt!='sumall' or plo!='compare':
                            pf.plot(which=whi,what=wha,option=opt,typeofplot=plo)
                for his in pf.listhist():
                    print(whi,wha,opt,his,"hist")
                    pf.hist(which=whi,what=wha,option=opt,typeofhist=his)
                for mo in pf.listmap():
                    if opt=='sumall':
                        continue
                    print(whi,wha,opt,'map',mo)
                    for lt in pf.listtile():
                        print(lt)
                        pf.map(which=whi,what=wha,option=opt,typeofmap=mo,tile=lt)
                for lo in pf.listoutput():
                    pf.get(which=whi,what=wha,option=opt,output=lo)

In [ ]:
# test daily with sumall for all db (should not diverge)
pf.setvis('matplotlib')
for who in pf.listwhom():
    pf.setwhom(who,reload=False)
    for wh in pf.listwhich():
        pf.plot(option='sumall',which=wh,what='daily')

In [ ]:
# Testing get
## outputs
for o in pf.listoutput():
    display(pf.get(when='01/01/2023',output=o))
    
## what
for wha in pf.listwhat():
    print(wha)
    display(pf.get(when='01/01/2023',what=wha))
    for op in pf.listoption():
        print(op)
        display(pf.get(when='01/01/2023',what=wha,option=op))
    print('all opt')
    display(pf.get(when='01/01/2023',what=wha,option=pf.listoption()[0:4] ))

# Archive against live

Every database is read from its Zenodo archive and from its live source (`pf.setlive(True)`), and the first and last date with data of each are compared. A live source more recent than its archive by over a month is flagged: a hint that the archive could be refreshed. A live source that cannot be read any more (gone upstream) is flagged as well.

The live read comes first, so that the archive read last leaves each database's local copy as the other cells expect it. This cell downloads every live source: it takes a while.

In [ ]:
import pandas as pd

def data_span(frame):
    """Return the first and last date a database has a value on."""
    values = [c for c in frame.columns if c not in ('date', 'where', 'code', 'geometry', 'from_db')]
    has = frame[values].notna().any(axis=1) & frame['from_db'].astype(bool)
    dates = pd.to_datetime(frame.loc[has, 'date'])
    return (dates.min().date(), dates.max().date()) if len(dates) else (None, None)

spans = []
for db in pf.listwhom():
    span = {'db': db}
    for live in (True, False):          # the archive last
        mode = 'live' if live else 'archive'
        try:
            pf.setlive(live)
            pf.setwhom(db, reload=True)
            span[mode+'_first'], span[mode+'_last'] = data_span(pf.getdatabase())
        except Exception as e:
            span[mode+'_first'], span[mode+'_last'] = None, None
            print(db, mode, 'unreadable:', e)
        finally:
            pf.setlive(False)
    spans.append(span)

spans = pd.DataFrame(spans)
spans['live_newer_days'] = (pd.to_datetime(spans['live_last']) - pd.to_datetime(spans['archive_last'])).dt.days
spans['flag'] = ''
spans.loc[spans['live_newer_days'] > 30, 'flag'] = 'live newer'
spans.loc[spans['live_last'].isna() & spans['archive_last'].notna(), 'flag'] = 'live unreadable'
# a live source known to be gone is read from the archive, with a warning
from pyvoa.jsondb_parser import MetaInfo
gone = spans['db'].map(lambda db: any(d.get('archiveonly') for d in MetaInfo().getcurrentmetadata(db)['datasets']))
spans.loc[gone, 'flag'] = 'live source gone'
spans